In [ ]:
!pip install nba_api

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Get LeBron's player information and seasons

In [ ]:
import pandas as pd
pd.options.display.max_columns = None

from nba_api.stats.endpoints import commonplayerinfo

player_info = commonplayerinfo.CommonPlayerInfo(player_id=2544)
lebron_season_df = player_info.available_seasons.get_data_frame()
lebron_season_df['SEASON_ID'] = lebron_season_df[lebron_season_df['SEASON_ID'].str.startswith('1') | lebron_season_df['SEASON_ID'].str.startswith('2')]
lebron_season_df = lebron_season_df[lebron_season_df['SEASON_ID'].notnull()]

# Let's get all the season ids to a list
seasons = lebron_season_df['SEASON_ID'].to_list()

# Get all games played

In [ ]:
from nba_api.stats.endpoints import playergamelog

all_games_df = pd.DataFrame()

for season in seasons:
    season_year = season[1:5]  # Extract the year part from SEASON_ID (e.g., '2003' from '12003')
    next_year = str(int(season_year) + 1)[-2:]  # Get the last two digits of the next year (e.g., '04')
    
    season_str = f"{season_year}-{next_year}"  # Format season as 'YYYY-YY' (e.g., '2003-04')
    
    game_log = playergamelog.PlayerGameLog(player_id=2544, season=season_str)
    game_df = game_log.get_data_frames()[0]
    all_games_df = pd.concat([all_games_df, game_df], ignore_index=True)

all_games_df

In [ ]:
all_games_df.columns

# Exploratory Data Analysis 

(kind of useless right now but whatever)

In [ ]:
# # Correlation matrix
# correlation_matrix = all_games_df.corr()

# # Plot a heatmap of the correlation matrix
# plt.figure(figsize=(20, 20))
# sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
# plt.title('Correlation Matrix')
# plt.show()

In [ ]:
df = pd.DataFrame()
df['PTS'] = all_games_df['PTS']
df['AST'] = all_games_df['AST']
df['REB'] = all_games_df['REB']

# Plot the distribution of the target variables
for i, col in enumerate(df.columns):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 3, i+1)
    sns.histplot(df[col], kde=True, color='green')
    plt.title('Distrubution of ' + col)

plt.tight_layout()
plt.show()

# Drop unnecessary columns

In [ ]:
cols_to_drop = ['SEASON_ID', 'Game_ID', 'VIDEO_AVAILABLE', 'WL']
all_games_df.drop(columns=cols_to_drop, inplace=True)

# Divide categorical and continuous columns

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

LE = LabelEncoder()
SS = StandardScaler()
categorical = ['GAME_DATE', 'MATCHUP']
continuous = ['MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA',
       'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF',
       'PTS', 'PLUS_MINUS']
for col in categorical:
    all_games_df[col] = LE.fit_transform(all_games_df[col])
for col in continuous:
    all_games_df[col] = SS.fit_transform(all_games_df[col].values.reshape(-1, 1))

In [ ]:
all_games_df.isnull().sum()

# Prepare the training and validation datasets

In [ ]:
from sklearn.model_selection import train_test_split

y = all_games_df[['PTS','REB','AST']]
X = all_games_df[['GAME_DATE', 'MATCHUP', 'Player_ID']]
X = pd.get_dummies(X, columns=['Player_ID'], drop_first=True)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=42)

# Training the different models

In [ ]:
# Create and train the multi-target regression model (ElasticNet)
multioutput_model = MultiOutputRegressor(
    ElasticNet(alpha=0.5, l1_ratio=0.5), n_jobs=5)
multioutput_model.fit(X_train, y_train)

# Create and train the decision tree regressor model
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train, y_train)

# Create and train the random forest regressor model
forest_model = RandomForestRegressor(n_estimators=100, random_state=42)
forest_model.fit(X_train, y_train)

# Predicting

In [ ]:
# Multioutput predictions
multioutput_preds = multioutput_model.predict(X_valid)
multioutput_mse = mean_squared_error(y_valid, multioutput_preds)
multioutput_mae = mean_absolute_error(y_valid, multioutput_preds)
# Tree predictions
tree_preds = tree_model.predict(X_valid)
tree_mse = mean_squared_error(y_valid, tree_preds)
tree_mae = mean_absolute_error(y_valid, tree_preds)
# Random forest predictions
forest_preds = forest_model.predict(X_valid)
forest_mse = mean_squared_error(y_valid, forest_preds)
forest_mae = mean_absolute_error(y_valid, forest_preds)

In [ ]:
# Print the performance metrics
print("Multioutput Model - Mean Squared Error:", multioutput_mse)
print("Multioutput Model - Mean Absolute Error:", multioutput_mae)
print("Decision Tree Model - Mean Squared Error:", tree_mse)
print("Decision Tree Model - Mean Absolute Error:", tree_mae)
print("Random Forest Model - Mean Squared Error:", forest_mse)
print("Random Forest Model - Mean Absolute Error:", forest_mae)

# Visualizing the performance of each model

In [ ]:
plt.figure(figsize=(10, 4))
models = ['Multioutput', 'Decision Tree', 'Random Forest']
mse_scores = [multioutput_mse, tree_mse, forest_mse]
mae_scores = [multioutput_mae, tree_mae, forest_mae]

plt.subplot(1, 2, 1)
plt.bar(models, mse_scores)
plt.xlabel('Models')
plt.ylabel('MSE')

plt.subplot(1, 2, 2)
plt.bar(models, mae_scores)
plt.xlabel('Models')
plt.ylabel('MAE')

plt.tight_layout()
plt.show()